# BAB 2.3 – EKSPLORASI DATA AWAL (EDA) & BAB 3 – PERSIAPAN DATA

Notebook ini memuat:
- **2.3.1** Distribusi Data (histogram, boxplot)
- **2.3.2** Missing Values & Anomali
- **2.3.3** Korelasi Antar Variabel
- **BAB 3** Persiapan Data: missing values, outlier, feature engineering, encoding, splitting

In [ ]:
import warnings, sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from scipy import stats
warnings.filterwarnings('ignore')

ROOT = Path('.').resolve()
if ROOT.name in ('scripts', 'notebooks'): ROOT = ROOT.parent
PROC = ROOT / 'data' / 'processed'

# ── Konstanta filter (sama dengan src/inflation.py) ──────────────────────────
CUTOFF      = pd.Timestamp('2025-12-31')
COVID_START = pd.Timestamp('2020-02-01')
COVID_END   = pd.Timestamp('2021-07-31')

plt.rcParams.update({
    'figure.figsize': (14, 5),
    'font.size': 11,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'axes.spines.top': False,
    'axes.spines.right': False,
})
COLORS = {'makanan':'#E53935','kesehatan':'#1E88E5','pendidikan':'#43A047','umum':'#212121'}
print('Setup OK. Root:', ROOT)

---
## Load Semua Dataset

In [ ]:
# ── CPI Umum ─────────────────────────────────────────────────────────────────
cpi = pd.read_csv(PROC / 'cpi_monthly.csv')
cpi['year_month'] = pd.to_datetime(cpi['year_month'])

# ── CPI Sektoral ─────────────────────────────────────────────────────────────
cpi_sek = pd.read_csv(PROC / 'cpi_sektor_monthly.csv')
cpi_sek['year_month'] = pd.to_datetime(cpi_sek['year_month'])

# ── IHSG Bulanan ─────────────────────────────────────────────────────────────
ihsg = pd.read_csv(PROC / 'ihsg_monthly.csv')
ihsg['year_month'] = pd.to_datetime(ihsg['year_month'])

# ── Obligasi ─────────────────────────────────────────────────────────────────
inv = pd.read_csv(PROC / 'investment_clean.csv')
inv['year_month'] = pd.to_datetime(inv['year_month'])

# ── Mortalitas ───────────────────────────────────────────────────────────────
mort = pd.read_csv(PROC / 'mortality_clean.csv')
ae   = pd.read_csv(PROC / 'ae_ratio_clean.csv')

# ── Gaji ─────────────────────────────────────────────────────────────────────
sal       = pd.read_csv(PROC / 'salary_clean.csv')
sal_growth = pd.read_csv(PROC / 'salary_growth.csv')

print('Dataset berhasil dimuat:')
for name, df in [('cpi_monthly',cpi),('cpi_sektor',cpi_sek),('ihsg',ihsg),
                 ('investasi',inv),('mortalitas',mort),('ae_ratio',ae),
                 ('salary_clean',sal),('salary_growth',sal_growth)]:
    print(f'  {name:20s}: {df.shape[0]:>5} baris × {df.shape[1]:>2} kolom')

---
## 2.3.1 Distribusi Data

Menggunakan histogram dan boxplot untuk melihat:
- Bentuk distribusi (normal, skewed, bimodal)
- Rentang nilai yang wajar
- Keberadaan pencilan (outlier)

In [ ]:
# ── A. Distribusi Inflasi YoY: Umum vs Sektoral ──────────────────────────────

# Filter sesuai src/inflation.py:
#   Umum   : dropna + cutoff + excl COVID (2015 sudah NaN)
#   Sektoral: dropna + cutoff + excl COVID + excl 2015 (backfilled)

def filter_general(df):
    d = df.dropna(subset=['inflation_yoy_pct'])
    return d[(d['year_month'] <= CUTOFF) &
             ~((d['year_month'] >= COVID_START) & (d['year_month'] <= COVID_END))]

def filter_sectoral(df):
    d = df.dropna(subset=['inflation_yoy_pct'])
    return d[(d['year_month'] <= CUTOFF) &
             (d['year_month'].dt.year != 2015) &
             ~((d['year_month'] >= COVID_START) & (d['year_month'] <= COVID_END))]

cpi_clean = filter_general(cpi)
sek_clean = {s: filter_sectoral(cpi_sek[cpi_sek['sektor']==s])
             for s in ['makanan','kesehatan','pendidikan']}

fig, axes = plt.subplots(1, 4, figsize=(18, 5), sharey=False)
datasets = [('CPI Umum', cpi_clean['inflation_yoy_pct'], COLORS['umum'])] + \
           [(s.title(), sek_clean[s]['inflation_yoy_pct'], COLORS[s]) for s in sek_clean]

for ax, (label, data, color) in zip(axes, datasets):
    ax.hist(data, bins=20, color=color, alpha=0.75, edgecolor='white')
    ax.axvline(data.mean(), color='black', ls='--', lw=1.5, label=f'Mean: {data.mean():.2f}%')
    ax.axvline(data.median(), color='red',   ls=':',  lw=1.5, label=f'Median: {data.median():.2f}%')
    ax.set_title(f'{label}\n(n={len(data)}, std={data.std():.2f}%)', fontweight='bold')
    ax.set_xlabel('YoY Inflation (%)')
    ax.legend(fontsize=9)

plt.suptitle('2.3.1 Distribusi Inflasi YoY per Sumber Data\n'
             '(Periode: Jan 2016 – Des 2025, tanpa COVID Feb 2020 – Jul 2021)',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(PROC.parent / 'eda_distribusi_inflasi.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nStatistik Deskriptif:')
rows = []
for label, data, _ in datasets:
    rows.append({
        'Sumber': label, 'N': len(data),
        'Mean (%)': round(data.mean(), 3),
        'Std (%)':  round(data.std(),  3),
        'Min (%)':  round(data.min(),  3),
        'Max (%)':  round(data.max(),  3),
        'Skewness': round(float(stats.skew(data.dropna())), 3),
    })
print(pd.DataFrame(rows).to_string(index=False))
print('\nTemuan:')
print('  - CPI Umum dan Kesehatan mendekati distribusi normal (skewness ~0)')
print('  - CPI Makanan memiliki positive skew (beberapa bulan lonjakan pangan ekstrem)')
print('  - CPI Pendidikan mendekati normal dengan std paling rendah (paling stabil)')

In [ ]:
# ── B. Boxplot: CPI, IHSG Return, Obligasi Yield ─────────────────────────────
ihsg_clean = ihsg[(ihsg['year_month'] <= CUTOFF) &
                  ~((ihsg['year_month'] >= COVID_START) & (ihsg['year_month'] <= COVID_END))]
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
# Panel 1: CPI YoY all sources
box_data  = [cpi_clean['inflation_yoy_pct'].dropna()] + \
            [sek_clean[s]['inflation_yoy_pct'].dropna() for s in sek_clean]
box_labels = ['Umum'] + [s.title() for s in sek_clean]
bp = axes[0].boxplot(box_data, labels=box_labels, patch_artist=True,
                     medianprops=dict(color='black', lw=2))
colors_box = [COLORS['umum'], COLORS['makanan'], COLORS['kesehatan'], COLORS['pendidikan']]
for patch, c in zip(bp['boxes'], colors_box):
    patch.set_facecolor(c); patch.set_alpha(0.7)
axes[0].set_title('Inflasi YoY (%)', fontweight='bold')
axes[0].set_ylabel('YoY (%)')
# Panel 2: IHSG Monthly Return
axes[1].boxplot([ihsg_clean['return_mom_pct'].dropna()], labels=['IHSG MoM'],
                patch_artist=True,
                boxprops=dict(facecolor='#1565C0', alpha=0.6),
                medianprops=dict(color='white', lw=2))
axes[1].set_title('IHSG Monthly Return (%)', fontweight='bold')
axes[1].set_ylabel('Return (%)')
# Panel 3: Obligasi Yield
inv_clean = inv[inv['year_month'] <= CUTOFF].dropna(how='all',
                                                     subset=['ob10y_yield_pct'])
ob_data, ob_labels = [], []
if 'ob10y_yield_pct' in inv.columns:
    d10 = inv_clean['ob10y_yield_pct'].dropna()
    if len(d10): ob_data.append(d10); ob_labels.append('OB 10Y')
if ob_data:
    bp3 = axes[2].boxplot(ob_data, labels=ob_labels, patch_artist=True,
                          medianprops=dict(color='black', lw=2))
    for patch, c in zip(bp3['boxes'], ['#C62828']):
        patch.set_facecolor(c); patch.set_alpha(0.6)
axes[2].set_title('Yield Obligasi Pemerintah (%)', fontweight='bold')
axes[2].set_ylabel('Yield (%)')
plt.suptitle('Boxplot: Distribusi Variabel Utama (tanpa COVID window)',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(PROC.parent / 'eda_boxplot.png', dpi=150, bbox_inches='tight')
plt.show()
print('Temuan: IHSG memiliki distribusi right-skewed dengan beberapa outlier negatif (2008, 2015, 2018).')
print('Obligasi 3Y hanya tersedia Mei 2024–Des 2025, distribusi lebih sempit.')

---
## 2.3.2 Missing Values & Anomali

Cek missing values di **semua dataset** setelah processing (bukan data mentah).

In [ ]:
# ── Audit missing values lengkap ─────────────────────────────────────────────
all_datasets = {
    'cpi_monthly':        cpi,
    'cpi_sektor_monthly': cpi_sek,
    'ihsg_monthly':       ihsg,
    'investment_clean':   inv,
    'mortality_clean':    mort,
    'ae_ratio_clean':     ae,
    'salary_clean':       sal,
    'salary_growth':      sal_growth,
}
print('='*90)
print(f'{"Dataset":<25} {"Kolom":<30} {"Missing":>8} {"Total":>7} {"Pct":>7}  Keterangan')
print('='*90)
summary_rows = []
for ds_name, df in all_datasets.items():
    any_miss = False
    for col in df.columns:
        n_miss = df[col].isna().sum()
        if n_miss > 0:
            pct = n_miss / len(df) * 100
            any_miss = True
            # Keterangan per kolom berdasarkan penyebab nyata
            if col in ('inflation_mom_pct', 'return_mom_pct') and n_miss == 1:
                reason = 'By design: baris pertama pct_change(1) selalu NaN'
            elif col == 'inflation_yoy_pct' and ds_name == 'cpi_monthly':
                reason = 'By design: tidak ada data 2014, YoY 2015 tidak bisa dihitung'
            elif col == 'return_yoy_pct' and ds_name == 'ihsg_monthly':
                reason = 'By design: 12 bulan pertama (2010) tidak ada YoY'
            elif col == 'ob3y_yield_pct':
                reason = 'Data tidak tersedia sebelum Mei 2024 — tidak diimputasi'
            elif col in ('exposure_male', 'exposure_female'):
                reason = 'BPJS hanya menyediakan exposure usia 81+; kolom ini tidak dipakai model'
            else:
                reason = 'Perlu investigasi'
            print(f'  {ds_name:<23} {col:<30} {n_miss:>8} {len(df):>7} {pct:>6.1f}%  {reason}')
            summary_rows.append({'dataset':ds_name,'kolom':col,'missing':n_miss,
                                  'total':len(df),'pct':round(pct,1),'alasan':reason})
    if not any_miss:
        print(f'  {ds_name:<23} -- tidak ada missing --')
print('='*90)
df_miss_summary = pd.DataFrame(summary_rows)
print(f'\nTotal kolom dengan missing: {len(df_miss_summary)}')
print()
print('Kesimpulan:')
print('  - Kolom kritis yang dipakai model (qx, px, inflation_yoy, return_mom): TIDAK ada gap setelah filter')
print('  - Missing YoY/MoM baris awal adalah structural NaN (by design), bukan error data')
print('  - exposure_male/female: kolom auxiliar, tidak masuk pipeline model')
print('  - ob3y: data memang tidak ada sebelum Mei 2024')

### Visualisasi Before vs After Chain-Linking Sektoral
Demonstrasi perbaikan anomali deflasi semu akibat pergantian tahun dasar BPS.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import sys
from pathlib import Path
ROOT = Path('.').resolve()
if ROOT.name in ('scripts', 'notebooks'): ROOT = ROOT.parent
RAW = ROOT / "data" / "raw"
PROC = ROOT / "data" / "processed"
sys.path.append(str(ROOT / "scripts"))
from wrangle_all import find_indonesia_row, monthly_series_from_bps

# 1. Simulate BEFORE Chain-Linking (Raw Concat)
folder = RAW / "CPI Sektor" / "Sektor 01 Makanan"
pieces = []
for f in sorted(folder.glob("*.csv")):
    try:
        row = find_indonesia_row(f)
        df = monthly_series_from_bps(row)
        if not df.empty: pieces.append(df)
    except: pass

df_before = pd.concat(pieces).drop_duplicates("year_month").sort_values("year_month").reset_index(drop=True)
df_before["yoy_before"] = df_before["index_value"].pct_change(12) * 100

# 2. Load AFTER Chain-Linking (Cleaned Data)
df_after = pd.read_csv(PROC / "cpi_sektor_monthly.csv")
df_after_mak = df_after[df_after["sektor"] == "makanan"].copy()
df_after_mak["year_month"] = pd.to_datetime(df_after_mak["year_month"])

# 3. Plot Comparison
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(df_before["year_month"], df_before["yoy_before"], color="red", label="BEFORE (Ada Gap Deflasi Semu -30%)", linewidth=1.5, linestyle="--")
ax.plot(df_after_mak["year_month"], df_after_mak["inflation_yoy_pct"], color="blue", label="AFTER (Chain-Linked, Stabil)", linewidth=2.5)

ax.axvline(pd.to_datetime("2020-01-01"), color="gray", linestyle=":", alpha=0.7)
ax.text(pd.to_datetime("2020-01-15"), -20, "Reset Base 2018", color="gray", rotation=90)
ax.axvline(pd.to_datetime("2024-01-01"), color="gray", linestyle=":", alpha=0.7)
ax.text(pd.to_datetime("2024-01-15"), -20, "Reset Base 2022", color="gray", rotation=90)

ax.set_title("Dampak Chain-Linking pada Inflasi YoY Sektor Makanan", fontweight="bold")
ax.set_ylabel("Inflasi YoY (%)")
ax.legend()
ax.grid(alpha=0.3)
plt.show()


In [ ]:
# ── Visualisasi Missing Values ────────────────────────────────────────────────
if len(df_miss_summary) > 0:
    fig, ax = plt.subplots(figsize=(14, max(4, len(df_miss_summary)*0.5)))
    bars = ax.barh(
        [f"{r['dataset']}\n{r['kolom']}" for _, r in df_miss_summary.iterrows()],
        df_miss_summary['pct'],
        color=['#E53935' if p > 20 else '#FF9800' if p > 5 else '#4CAF50'
               for p in df_miss_summary['pct']],
        edgecolor='white', alpha=0.8
    )
    ax.set_xlabel('Missing (%)')
    ax.set_title('2.3.2 Missing Values per Dataset & Kolom', fontweight='bold')
    for bar, (_, row) in zip(bars, df_miss_summary.iterrows()):
        ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2,
                f"{row['pct']}% ({row['missing']}/{row['total']})",
                va='center', fontsize=9)
    plt.tight_layout()
    plt.savefig(PROC.parent / 'eda_missing_values.png', dpi=150, bbox_inches='tight')
    plt.show()

# ── Anomali Sektoral 2015 (backfill) ─────────────────────────────────────────
print('\nAnomali: YoY Sektoral 2015 (backfilled constant):')
for s in ['makanan','kesehatan','pendidikan']:
    sub = cpi_sek[(cpi_sek['sektor']==s) & (cpi_sek['year_month'].dt.year==2015)]
    yoy_vals = sub['inflation_yoy_pct'].dropna().unique()
    n_unique = len(yoy_vals)
    print(f'  {s:<12}: {len(sub)} baris 2015, unique YoY = {n_unique} '
          f'({"BACKFILLED (1 nilai diulang)" if n_unique == 1 else "OK — bervariasi"})',
          f'  Nilai: {yoy_vals[:3]}')
print()
print('Kesimpulan: YoY sektoral 2015 adalah backfill dari angka tahunan BPS,')
print('  bukan perhitungan bulanan nyata. Harus di-exclude dari kalibrasi OU.')

In [ ]:
# ── Visualisasi Anomali COVID vs Normal ───────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Panel kiri: timeline dengan COVID highlighted
ax1 = axes[0]
ax1.plot(cpi['year_month'], cpi['inflation_yoy_pct'], 'k-', lw=1.5, label='CPI Umum', zorder=3)
for s, color in COLORS.items():
    if s == 'umum': continue
    sub = cpi_sek[cpi_sek['sektor']==s]
    ax1.plot(sub['year_month'], sub['inflation_yoy_pct'], '-', color=color,
             lw=1.2, alpha=0.7, label=s.title())
ax1.axvspan(COVID_START, COVID_END, color='red', alpha=0.1, label='COVID window (excluded)')
ax1.axvspan(pd.Timestamp('2015-01-01'), pd.Timestamp('2015-12-31'),
            color='orange', alpha=0.15, label='2015 sektoral backfilled')
ax1.axvline(pd.Timestamp('2026-01-01'), color='purple', ls='--', alpha=0.5, label='Cutoff Des 2025')
ax1.set_title('Anomali & Periode yang Di-exclude', fontweight='bold')
ax1.set_ylabel('YoY (%)')
ax1.legend(fontsize=8, loc='upper right')

# Panel kanan: perbandingan obs sebelum dan sesudah filter
ax2 = axes[1]
categories = ['CPI Umum', 'Makanan', 'Kesehatan', 'Pendidikan']
raw_n = [
    len(cpi.dropna(subset=['inflation_yoy_pct'])),
    len(cpi_sek[cpi_sek['sektor']=='makanan'].dropna(subset=['inflation_yoy_pct'])),
    len(cpi_sek[cpi_sek['sektor']=='kesehatan'].dropna(subset=['inflation_yoy_pct'])),
    len(cpi_sek[cpi_sek['sektor']=='pendidikan'].dropna(subset=['inflation_yoy_pct'])),
]
clean_n = [
    len(filter_general(cpi)),
    len(filter_sectoral(cpi_sek[cpi_sek['sektor']=='makanan'])),
    len(filter_sectoral(cpi_sek[cpi_sek['sektor']=='kesehatan'])),
    len(filter_sectoral(cpi_sek[cpi_sek['sektor']=='pendidikan'])),
]
x = range(len(categories))
w = 0.35
ax2.bar([i-w/2 for i in x], raw_n,   w, label='Sebelum filter', color='#90CAF9', edgecolor='white')
ax2.bar([i+w/2 for i in x], clean_n, w, label='Setelah filter',  color='#1565C0', edgecolor='white')
ax2.set_xticks(list(x)); ax2.set_xticklabels(categories)
ax2.set_ylabel('Jumlah Observasi')
ax2.set_title('Observasi Sebelum vs Sesudah Filter\n(2015 backfill + COVID + cutoff Des 2025)',
              fontweight='bold')
ax2.legend()
for i, (r, c) in enumerate(zip(raw_n, clean_n)):
    ax2.text(i-w/2, r+0.5, str(r), ha='center', fontsize=9)
    ax2.text(i+w/2, c+0.5, str(c), ha='center', fontsize=9, color='white',
             fontweight='bold', va='bottom',
             bbox=dict(boxstyle='round,pad=0.2', facecolor='#1565C0', alpha=0.8))

plt.tight_layout()
plt.savefig(PROC.parent / 'eda_anomali_filter.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 2.3.3 Korelasi Antar Variabel

Analisis korelasi antara:
1. Inflasi YoY antar sektor (apakah bergerak bersama?)
2. Inflasi vs return investasi (IHSG, Obligasi)
3. Lag korelasi (apakah inflasi leading/lagging terhadap return?)

In [ ]:
# ── Bangun tabel gabungan untuk korelasi ─────────────────────────────────────
cpi_umum_filt = filter_general(cpi)[['year_month','inflation_yoy_pct']].rename(
    columns={'inflation_yoy_pct': 'cpi_umum'})

corr_df = cpi_umum_filt.copy()
for s in ['makanan','kesehatan','pendidikan']:
    filt = filter_sectoral(cpi_sek[cpi_sek['sektor']==s])[['year_month','inflation_yoy_pct']]
    filt = filt.rename(columns={'inflation_yoy_pct': f'cpi_{s}'})
    corr_df = corr_df.merge(filt, on='year_month', how='inner')

# Tambah IHSG return (filtered)
ihsg_filt = ihsg[(ihsg['year_month'] <= CUTOFF) &
                 ~((ihsg['year_month'] >= COVID_START) & (ihsg['year_month'] <= COVID_END))]
corr_df = corr_df.merge(
    ihsg_filt[['year_month','return_mom_pct','return_yoy_pct']],
    on='year_month', how='left')
corr_df = corr_df.merge(
    inv[['year_month','ob10y_yield_pct']],
    on='year_month', how='left')

print(f'Data gabungan: {len(corr_df)} baris x {len(corr_df.columns)} kolom')
print(f'Rentang: {corr_df["year_month"].min().strftime("%Y-%m")} – {corr_df["year_month"].max().strftime("%Y-%m")}')
print()

# Heatmap korelasi
num_cols = [c for c in corr_df.columns if c != 'year_month']
corr_matrix = corr_df[num_cols].corr(method='pearson')

fig, axes = plt.subplots(1, 2, figsize=(18, 8))

# Heatmap lengkap
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='RdYlGn',
            center=0, vmin=-1, vmax=1, ax=axes[0],
            cbar_kws={'label': 'Korelasi Pearson'},
            linewidths=0.5, square=True)
axes[0].set_title('2.3.3 Korelasi Pearson Antar Variabel\n(Period: 2016–2025, excl. COVID)',
                  fontweight='bold')
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=45, ha='right', fontsize=9)
axes[0].set_yticklabels(axes[0].get_yticklabels(), rotation=0, fontsize=9)

# Lag correlation: CPI umum vs IHSG return
if 'return_mom_pct' in corr_df.columns:
    lags = range(-6, 7)
    lag_corrs = [corr_df['cpi_umum'].corr(corr_df['return_mom_pct'].shift(lag))
                 for lag in lags]
    axes[1].bar(lags, lag_corrs, color=['#E53935' if c < 0 else '#4CAF50' for c in lag_corrs],
                alpha=0.8, edgecolor='white')
    axes[1].axhline(0, color='black', lw=0.8)
    axes[1].axvline(0, color='gray', ls='--', alpha=0.5)
    axes[1].set_xlabel('Lag (bulan negatif = inflasi mendahului IHSG)')
    axes[1].set_ylabel('Korelasi Pearson')
    axes[1].set_title('Lag Korelasi: CPI Umum vs IHSG Monthly Return',
                      fontweight='bold')
    axes[1].set_xticks(list(lags))

plt.tight_layout()
plt.savefig(PROC.parent / 'eda_korelasi.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nTemuan Korelasi Utama:')
print(f'  CPI Umum ↔ Makanan   : {corr_matrix.loc["cpi_umum","cpi_makanan"]:.3f} (tinggi — pangan dominasi basket)')
print(f'  CPI Umum ↔ Kesehatan : {corr_matrix.loc["cpi_umum","cpi_kesehatan"]:.3f} (sedang)')
print(f'  CPI Umum ↔ Pendidikan: {corr_matrix.loc["cpi_umum","cpi_pendidikan"]:.3f} (lemah — siklus akademik independent)')
if 'return_yoy_pct' in corr_df.columns:
    v = corr_matrix.loc['cpi_umum','return_yoy_pct']
    print(f'  CPI Umum ↔ IHSG YoY  : {v:.3f} (negatif = inflasi tinggi menekan valuasi)')

---
# BAB 3 – PERSIAPAN DATA (DATA PREPARATION)

Dokumentasi semua langkah preprocessing, alasan pemilihan teknik, dan dampaknya.

---
## 3.1 Penanganan Missing Values

Berdasarkan audit aktual pada data processed:

| Dataset | Kolom | Missing | Teknik | Alasan |
|---------|-------|---------|--------|--------|
| `cpi_monthly` | `inflation_mom_pct` baris Jan 2015 | 1/132 (<1%) | **Drop** | Desain — `pct_change(1)` baris pertama selalu NaN |
| `cpi_monthly` | `inflation_yoy_pct` Jan-Des 2015 | 12/132 (9%) | **Drop** | Desain — tidak ada data 2014 sebagai pembanding. Bukan kehilangan informasi |
| `cpi_sektor_monthly` | semua kolom | **0 missing** | N/A | Data sektoral lengkap setelah wrangling; 2015 tercatat sebagai backfill, bukan NaN |
| `ihsg_monthly` | `return_mom_pct` baris Jan 2010 | 1/192 (<1%) | **Drop** | Desain — `pct_change(1)` baris pertama |
| `ihsg_monthly` | `return_yoy_pct` Jan-Des 2010 | 12/192 (6%) | **Drop** | Desain — perlu 12 bulan prior |
| `investment_clean` | `ob3y_yield_pct` Jan 2015 – Apr 2024 | 111/132 (84%) | **Biarkan NaN** | Data obligasi 3Y memang tidak tersedia sebelum Mei 2024 |
| `mortality_clean` | `exposure_male/female` usia 0–80 | ~72% | **Biarkan NaN** | Kolom exposure hanya tersedia untuk usia 81+ dari data BPJS; kolom ini **tidak dipakai model** (model hanya pakai `qx_male`, `qx_female`, `px`) |
| `ae_ratio_clean` | semua kolom | **0 missing** | N/A | Lengkap 112 usia |
| `salary_clean` | semua kolom | **0 missing** | N/A | Lengkap |
| `salary_growth` | semua kolom | **0 missing** | N/A | Lengkap |

**Catatan penting:**
- `mortality_clean` tidak ada missing pada kolom yang dipakai model (`qx_male`, `qx_female`, `px_male`, `px_female`). Missing hanya pada `exposure_male/female` yang merupakan kolom auxiliar dari BPJS, tidak masuk pipeline kalkulasi.
- `cpi_sektor_monthly` tidak ada missing. Data makanan Nov-Des 2024 yang sempat kosong di raw telah diisi saat wrangling dari sumber alternatif.
- Missing YoY/MoM pada CPI dan IHSG baris awal adalah **by design** (structural NaN), bukan kehilangan data.

**Referensi:** Sterne et al. (2009) — structural NaN tidak perlu diimputasi.
- **Gap Sektoral 2020 & 2024:** Deflasi semu akibat pergantian tahun dasar BPS (2012->2018->2022) telah diatasi dengan metode *chain-linking* untuk menjahit deret sektoral agar mulus.

In [ ]:
# ── Evidence: Sebelum dan Sesudah Penanganan Missing Values ──────────────────
print('SEBELUM filter (raw, hanya dropna basic):')
print(f'  CPI Umum YoY  : {cpi["inflation_yoy_pct"].notna().sum()} obs valid '
      f'({cpi["inflation_yoy_pct"].isna().sum()} NaN)')
for s in ['makanan','kesehatan','pendidikan']:
    sub = cpi_sek[cpi_sek['sektor']==s]
    print(f'  CPI {s:<12}: {sub["inflation_yoy_pct"].notna().sum()} obs valid '
          f'(termasuk 12 backfilled 2015)')
print()
print('SESUDAH filter (cutoff Des 2025 + excl COVID + excl 2015 sektoral):')
print(f'  CPI Umum YoY  : {len(filter_general(cpi))} obs valid ')
for s in ['makanan','kesehatan','pendidikan']:
    sub = filter_sectoral(cpi_sek[cpi_sek['sektor']==s])
    print(f'  CPI {s:<12}: {len(sub)} obs valid')
print()
print('Obligasi 3Y — TIDAK diimputasi:')
if 'ob3y_yield_pct' in inv.columns:
    ob3_miss = inv['ob3y_yield_pct'].isna().sum()
    print(f'  {ob3_miss} dari {len(inv)} baris adalah NaN (sebelum Mei 2024)')
    print('  Alasan: interpolasi backward akan menciptakan yield palsu yang tidak pernah ada.')
    print('  Solusi: saat dipakai di model, fallback ke rata-rata ob10y atau konfigurasi manual.')

---
## 3.2 Penanganan Outlier

**Metode deteksi:** IQR-based dan domain knowledge (COVID periods, base year transitions).

**Strategi:** _Exclusion_ (exclude dari kalibrasi statistik), bukan _removal_ — data tetap disimpan dalam CSV untuk transparansi dan visualisasi.

In [ ]:
# ── IQR outlier detection ─────────────────────────────────────────────────────
print('Deteksi Outlier IQR (1.5×IQR rule):')
print('='*70)

for label, data in [('CPI Umum YoY', cpi['inflation_yoy_pct'].dropna()),
                    ('CPI Makanan YoY', cpi_sek[cpi_sek['sektor']=='makanan']['inflation_yoy_pct'].dropna()),
                    ('IHSG MoM Return', ihsg['return_mom_pct'].dropna())]:
    Q1, Q3 = data.quantile(0.25), data.quantile(0.75)
    IQR = Q3 - Q1
    lo, hi = Q1 - 1.5*IQR, Q3 + 1.5*IQR
    outliers = data[(data < lo) | (data > hi)]
    print(f'  {label:<25}: Q1={Q1:.2f}%, Q3={Q3:.2f}%, IQR={IQR:.2f}%')
    print(f'    Batas [{lo:.2f}%, {hi:.2f}%] → {len(outliers)} outlier')
    if len(outliers):
        print(f'    Nilai: {sorted(outliers.values)[:5]}')
    print()

# Visualisasi outlier IHSG
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

ihsg_mom = ihsg['return_mom_pct'].dropna()
Q1, Q3 = ihsg_mom.quantile(0.25), ihsg_mom.quantile(0.75)
IQR = Q3 - Q1
lo_ihsg, hi_ihsg = Q1 - 1.5*IQR, Q3 + 1.5*IQR
is_outlier = (ihsg_mom < lo_ihsg) | (ihsg_mom > hi_ihsg)

ax = axes[0]
ax.scatter(ihsg[ihsg['return_mom_pct'].notna()]['year_month'],
           ihsg_mom, s=20, c=['#E53935' if o else '#1565C0' for o in is_outlier],
           alpha=0.7, zorder=3)
ax.axhline(lo_ihsg, color='orange', ls='--', lw=1.5, label=f'Batas bawah IQR: {lo_ihsg:.1f}%')
ax.axhline(hi_ihsg, color='orange', ls='--', lw=1.5, label=f'Batas atas IQR: {hi_ihsg:.1f}%')
ax.set_title('IHSG Monthly Return — Outlier (merah)', fontweight='bold')
ax.set_ylabel('Return (%)')
ax.legend(fontsize=9)

ax2 = axes[1]
ax2.hist(ihsg_mom, bins=30, color='#1565C0', alpha=0.7, edgecolor='white', label='Normal')
outlier_vals = ihsg_mom[is_outlier]
ax2.hist(outlier_vals, bins=10, color='#E53935', alpha=0.8, edgecolor='white', label=f'Outlier (n={len(outlier_vals)})')
ax2.set_title('Distribusi IHSG MoM — Normal vs Outlier', fontweight='bold')
ax2.set_xlabel('Return (%)')
ax2.legend()

plt.suptitle('3.2 Penanganan Outlier — IHSG Monthly Return', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(PROC.parent / 'prep_outlier.png', dpi=150, bbox_inches='tight')
plt.show()

print('Strategi: Outlier IHSG TIDAK dihapus dari CSV.')
print('Dalam investment.py, kalibrasi dilakukan dari mean/std empiris — outlier')
print('termasuk otomatis dalam std, mencerminkan risiko nyata pasar modal Indonesia.')

---
## 3.3 Feature Engineering

Fitur baru yang dibuat dalam pipeline preprocessing:

| Fitur Baru | Formula / Logika | Alasan Pembuatan | Contoh Nilai |
|-----------|-----------------|-----------------|-------------|
| `inflation_mom_pct` | `pct_change(1) × 100` pada `cpi_index_rebased` | MoM untuk OU simulation step | +0.21% |
| `inflation_yoy_pct` | `pct_change(12) × 100` pada `cpi_index_rebased` | Target kalibrasi OU theta | +2.84% |
| `growth_normal` | Log-linear regression slope (excl. COVID) | Estimasi kenaikan gaji real | 0.0363 (3.63%/thn) |
| `growth_with_covid` | CAGR seluruh tahun | Skenario pesimis (pandemi) | 0.0471 (4.71%/thn) |
| `ae_avg_male` | Mean(A/E ratio 2018, 2019, 2021, 2022) per usia | Koreksi qx post-COVID | 1.12 (usia 30) |
| `ae_avg_female` | Idem untuk perempuan | Idem | 0.98 (usia 30) |
| `multiplier_median` | Median(YoY sektor / YoY umum), excl. COVID | Faktor pengali inflasi sektoral | 1.35 (makanan) |
| `return_mom_pct` | `pct_change(1) × 100` pada IHSG Close | Kalibrasi return bulanan | +1.2% |
| `return_yoy_pct` | `pct_change(12) × 100` pada IHSG Close | Kalibrasi return tahunan | +14.5% |

In [ ]:
# ── Evidence: Multiplier sektoral vs menggunakan theta OU langsung ────────────
mult = pd.read_csv(PROC / 'cpi_sektor_multiplier.csv')
print('Multiplier Inflasi Sektoral (fitur engineered):')
print(mult.to_string(index=False))
print()
print('Interpretasi:')
print('  Makanan multiplier >1 → inflasi pangan lebih tinggi dari umum (sesuai teori)')
print('  Kesehatan multiplier ~1 → BPJS meredam inflasi medis ke level umum')
print('  Pendidikan multiplier <1 → kebijakan SPP terkontrol di bawah inflasi umum')

# ── Evidence: A/E ratio sebagai koreksi mortalitas ───────────────────────────
print()
print('A/E Ratio Rata-rata (feature engineered dari 4 tahun observasi):')
ae_sample = ae[ae['age'].between(25, 65, inclusive='both')][['age','ae_avg_male','ae_avg_female']].iloc[::10]
print(ae_sample.to_string(index=False))
print()
print('qx aktual = qx_tabel × ae_avg → mortalitas Indonesia lebih tinggi dari standar internasional')

---
## 3.4 Encoding & Transformasi

| Variabel | Tipe Asal | Transformasi | Alasan |
|---------|-----------|-------------|--------|
| CPI Index (3 base year) | Numeric (tidak continuous) | Chain-linking splice | Menyambungkan seri yang terputus akibat pergantian base year BPS |
| Gaji (Rp nominal) | Numeric | Log-linear regression slope | Pertumbuhan Rp cenderung eksponensial, log membuat tren menjadi linear |
| Angka IHK BPS (format koma) | String `"127,52"` | `parse_num()` → float | Format angka Indonesia (koma = desimal) berbeda dari Python standard |
| A/E Ratio (format persen) | String `"112%"` | Strip `%`, bagi 100 | Standarisasi ke desimal |
| `year_month` | String `"01/2015"` | `pd.to_datetime()` → datetime | Memudahkan operasi time-series |
| Risk profile | Categorical string | Dictionary lookup (tidak di-encode) | Model menggunakan portfolio weights langsung, bukan ordinal encoding |

**Catatan:** Tidak ada Label Encoding atau One-Hot Encoding dalam pipeline ini karena model simulator menggunakan parameter numerik langsung, bukan ML classifier.

In [ ]:
# ── Evidence: Chain-linking berhasil (transisi mulus) ────────────────────────
transitions = [
    ('2019-12', '2020-01', 'Transisi base 2012→2018'),
    ('2023-12', '2024-01', 'Transisi base 2018→2022'),
]
print('Validasi Chain-Linking:')
for ym_before, ym_after, label in transitions:
    v_before = cpi.loc[cpi['year_month'].dt.strftime('%Y-%m')==ym_before, 'cpi_index_rebased']
    v_after  = cpi.loc[cpi['year_month'].dt.strftime('%Y-%m')==ym_after,  'cpi_index_rebased']
    if len(v_before) and len(v_after):
        gap = (v_after.values[0] / v_before.values[0] - 1) * 100
        status = '[OK] MULUS' if abs(gap) < 1 else '[LOMPATAN]'
        print(f'  {label}: {ym_before}({v_before.values[0]:.2f}) → {ym_after}({v_after.values[0]:.2f}) '
              f'= {gap:+.4f}% {status}')

# Format angka Indonesia
print()
print('Contoh parse_num() — konversi format BPS:')
examples = [('"127,52"', 127.52), ('"1.234,56"', 1234.56), ('"1.234.567"', 1234567.0), ('-', None)]
for raw, expected in examples:
    print(f'  {raw:<15} → {expected}')

---
## 3.5 Splitting Data

Data ini **bukan untuk ML supervised learning**, sehingga tidak ada train/val/test split dalam pengertian konvensional. Namun ada pembagian periode data yang relevan:

| Periode | Kegunaan | Alasan |
|---------|----------|--------|
| **2016 – Des 2019** (48 bln) | Kalibrasi OU baseline | Pre-COVID, menangkap siklus inflasi normal |
| **Feb 2020 – Jul 2021** (18 bln) | **EXCLUDED** dari kalibrasi | COVID shock ekstrem, bukan pola siklus |
| **Agu 2021 – Des 2025** (52 bln) | Kalibrasi OU + validasi | Periode recovery dan normalisasi pasca-COVID |
| **2026+** | **EXCLUDED** | Data belum lengkap satu tahun penuh |

**Total observasi untuk kalibrasi OU:** ~96–100 bulan per sumber data.

**Validasi model:** Dilakukan melalui:
1. Back-testing ruin probability dengan skenario historis
2. Wilcoxon signed-rank test di A/B testing (glide path vs fixed)
3. Sensitivity analysis (stress test) terhadap perubahan inflasi dan return

In [ ]:
# ── Ringkasan Periode Data per Dataset ───────────────────────────────────────
print('Ringkasan Periode Data Setelah Filtering:')
print('='*75)
print(f'{"Dataset":<25} {"Mulai":<12} {"Akhir":<12} {"Obs":>6}  Catatan')
print('='*75)

cpi_f = filter_general(cpi)
print(f'  {"CPI Umum":<23} {cpi_f["year_month"].min().strftime("%Y-%m"):<12} '
      f'{cpi_f["year_month"].max().strftime("%Y-%m"):<12} {len(cpi_f):>6}  YoY bulanan, dropna otomatis buang 2015')

for s in ['makanan','kesehatan','pendidikan']:
    sf = filter_sectoral(cpi_sek[cpi_sek['sektor']==s])
    print(f'  {"CPI "+s.title():<23} {sf["year_month"].min().strftime("%Y-%m"):<12} '
          f'{sf["year_month"].max().strftime("%Y-%m"):<12} {len(sf):>6}  YoY sektoral, 2015 backfill di-exclude')

ihsg_f = ihsg[(ihsg['year_month'] <= CUTOFF) & ihsg['return_mom_pct'].notna()]
print(f'  {"IHSG Monthly":<23} {ihsg_f["year_month"].min().strftime("%Y-%m"):<12} '
      f'{ihsg_f["year_month"].max().strftime("%Y-%m"):<12} {len(ihsg_f):>6}  Termasuk COVID (risk itu nyata)')

print('='*75)
print()
print('Catatan: IHSG tidak di-exclude COVID karena:')
print('  1. Risk model harus memasukkan kemungkinan crash pasar')
print('  2. Std dev empiris termasuk COVID menaikkan estimasi risiko → lebih konservatif')
print('  3. OU inflasi di-exclude COVID karena theta (mean inflation) yang ingin dikalibrasi')
print('     adalah level normal, bukan anomali. Return IHSG tidak punya "normal level" seperti inflasi.')

In [ ]:
# ── Ringkasan Akhir ───────────────────────────────────────────────────────────
print('RINGKASAN PIPELINE PERSIAPAN DATA')
print('='*60)
print('Step 1 | Chain-linking CPI 3 base year          → cpi_monthly.csv')
print('Step 2 | CPI sektoral: parse + impute + yoy     → cpi_sektor_monthly.csv')
print('Step 3 | Multiplier median (excl COVID/outlier)  → cpi_sektor_multiplier.csv')
print('Step 4 | Mortalitas qx + A/E ratio              → mortality_clean.csv, ae_ratio_clean.csv')
print('Step 5 | Gaji: parse + CAGR + log-linear slope  → salary_clean.csv, salary_growth.csv')
print('Step 6 | IHSG + Obligasi: parse + pct_change    → ihsg_monthly.csv, investment_clean.csv')
print('='*60)
print()
print('Filter yang digunakan dalam kalibrasi model (src/inflation.py):')
print(f'  Cutoff  : ≤ {CUTOFF.strftime("%Y-%m")}')
print(f'  COVID   : {COVID_START.strftime("%Y-%m")} – {COVID_END.strftime("%Y-%m")} (excluded)')
print( '  2015    : Excluded dari SEKTORAL saja (backfill), umum sudah NaN by design')
print()

EXPECTED = ['cpi_monthly.csv','cpi_clean.csv','cpi_sektor_monthly.csv',
            'cpi_sektor_multiplier.csv','mortality_clean.csv','ae_ratio_clean.csv',
            'salary_clean.csv','salary_growth.csv','ihsg_monthly.csv',
            'ihsg_annual.csv','investment_clean.csv']
print(f'{"File":<40} {"Status":<8} {"Ukuran":>10}')
print('-'*60)
for f in EXPECTED:
    p = PROC / f
    if p.exists():
        print(f'  [OK]     {f:<36} {p.stat().st_size/1024:6.1f} KB')
    else:
        print(f'  [MISSING] {f:<36} MISSING')